# Skeptic Engine — Statistical Artifact Detection Demo

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19238786.svg)](https://doi.org/10.5281/zenodo.19238786)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

This notebook demonstrates the Skeptic Engine toolkit for detecting non-physical statistical artifacts in single-cell RNA-seq count matrices.

**What it does:** Extracts Benford digit features + cell-level structural features, trains an Isolation Forest on reference data, and produces a fusion artifact risk score.

**What it does NOT do:** It does not claim to detect fraud. Elevated scores indicate anomalous statistical patterns that warrant expert review.

**Published:** https://doi.org/10.5281/zenodo.19238786

---

## 1. Setup — Install dependencies and clone toolkit

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib
!git clone --depth 1 https://github.com/sergeeey/Skeptic-Engine-2026-.git skeptic_engine
import sys
sys.path.insert(0, 'skeptic_engine/experiments/h24_benford_scrna')

## 2. Generate toy datasets — Clean and fabricated count matrices

In [ ]:
import numpy as np

rng = np.random.default_rng(2026)
n_cells, n_genes = 500, 1000

# Simulate real scRNA-seq-like data: sparse, right-skewed integers
# Most values 0, some 1-5, few 10-100
real_matrix = np.zeros((n_cells, n_genes), dtype=np.int64)
for i in range(n_cells):
    n_expressed = rng.integers(50, 300)
    genes_on = rng.choice(n_genes, size=n_expressed, replace=False)
    counts = rng.negative_binomial(n=1, p=0.5, size=n_expressed)
    real_matrix[i, genes_on] = np.clip(counts, 1, 200)

print(f'Real matrix: {real_matrix.shape}')
print(f'Nonzero fraction: {(real_matrix > 0).mean():.4f}')
print(f'Value range: [{real_matrix.min()}, {real_matrix.max()}]')
print(f'Median nonzero: {np.median(real_matrix[real_matrix > 0])}')

In [ ]:
# Fabricate: per-gene NB generation (preserves mean/variance, breaks digit patterns)
fake_matrix = np.zeros_like(real_matrix)
for j in range(n_genes):
    col = real_matrix[:, j].astype(np.float64)
    mean_val = col.mean()
    var_val = col.var()
    if var_val > mean_val and mean_val > 0.01:
        n_param = max(mean_val**2 / (var_val - mean_val), 0.1)
        p_param = min(max(mean_val / var_val, 0.01), 0.99)
        fake_matrix[:, j] = rng.negative_binomial(n=n_param, p=p_param, size=n_cells)
    elif mean_val > 0.01:
        fake_matrix[:, j] = rng.poisson(lam=mean_val, size=n_cells)

fake_matrix = fake_matrix.astype(np.int64)
print(f'Fake matrix: {fake_matrix.shape}')
print(f'Nonzero fraction: {(fake_matrix > 0).mean():.4f}')

## 3. Extract features — Benford digits + cell-level statistics

In [ ]:
from digit_features import extract_features_per_sample
from isolation_forest import cell_level_features, train_isolation_forest, score_anomalies

# Benford features (21 per cell)
real_benford = extract_features_per_sample(real_matrix)
fake_benford = extract_features_per_sample(fake_matrix)

# Cell-level features (8 per cell)
real_cell = cell_level_features(real_matrix)
fake_cell = cell_level_features(fake_matrix)

# Isolation Forest (trained on real only)
if_model = train_isolation_forest(real_cell)
real_if = score_anomalies(if_model, real_cell)
fake_if = score_anomalies(if_model, fake_cell)

# Fusion features
real_fusion = np.hstack([real_benford, real_cell, real_if.reshape(-1, 1)])
fake_fusion = np.hstack([fake_benford, fake_cell, fake_if.reshape(-1, 1)])

print(f'Features per cell: {real_fusion.shape[1]} (Benford: 21, Cell: 8, IF: 1)')

## 4. Train classifier and evaluate

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib.pyplot as plt

X = np.vstack([real_fusion, fake_fusion])
y = np.concatenate([np.zeros(n_cells), np.ones(n_cells)])

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y))

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X[train_idx], y[train_idx])

y_prob = rf.predict_proba(X[test_idx])[:, 1]
auc = roc_auc_score(y[test_idx], y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(rf, X[test_idx], y[test_idx], ax=ax, name='Fusion (30 features)')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_title(f'Artifact Detection: AUC = {auc:.4f}')
plt.tight_layout()
plt.show()

print(f'\nAUC-ROC: {auc:.4f}')
print('Interpretation: 1.0 = perfect separation, 0.5 = random chance')

## 5. Key finding — Benford digit distributions differ

Real scRNA-seq UMI counts do NOT follow Benford's Law (first digit ≈ 0.74, not 0.30).
This means **Benford compliance is itself suspicious** in scRNA-seq data.

In [ ]:
from digit_features import BENFORD_FIRST

digits = np.arange(1, 10)
fig, ax = plt.subplots(figsize=(8, 5))
w = 0.25
ax.bar(digits - w, real_benford[:, :9].mean(axis=0), w, label='Real', color='#26A69A')
ax.bar(digits, fake_benford[:, :9].mean(axis=0), w, label='Fabricated (NB)', color='#EC407A')
ax.bar(digits + w, BENFORD_FIRST, w, label='Benford expected', color='#9E9E9E', alpha=0.6)
ax.set_xlabel('First Digit')
ax.set_ylabel('Mean Frequency')
ax.set_title('First-Digit Distributions: Real vs Fabricated vs Benford')
ax.legend()
ax.set_xticks(digits)
plt.tight_layout()
plt.show()

## Summary

- **Within-dataset** artifact detection achieves AUC 0.86-1.00 on real scRNA-seq data (PBMC3k, Kang2018)
- **Cross-dataset** generalization is limited for sophisticated artifact types
- **Benford compliance is suspicious** in scRNA-seq (novel observation)
- The tool is designed as a **per-dataset screening step**, not a universal detector

For full results, see `REPORT.md` in the repository.

---
*Skeptic Engine — Statistical Artifact Detection for Scientific Data Integrity*